In [4]:
# 读取/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/minimized中的所有pdb，采用py_contact_ms.py计算cms
import os
import subprocess
import pandas as pd
import sys
from pyrosetta import init, pose_from_file
from py_contact_ms import partition_pose, calculate_contact_ms, calculate_maximum_possible_contact_ms
init('-mute all')

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2026 [Rosetta PyRosetta4.Release.python310.ubuntu 2026.03+releasequarterly.5e498f1409c68ade56c8ce5842bf79e1b02e8db4 2026-01-13T13:24:11] retrieved from: http://www.pyrosetta.org


In [5]:
cms_results = []
for file in os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/minimized"):
    if file.endswith(".pdb"):
        pdb = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/minimized", file)
        # 计算cms

        pose = pose_from_file(pdb)

        binder_xyz, binder_radii, target_xyz, target_radii= partition_pose(pose)
        cms, per_target_atom_cms, calc = calculate_contact_ms(binder_xyz, binder_radii, target_xyz, target_radii)

        # print('CMS: ', cms)


        d0    = calc.run.dots[0]
        dots0 = d0.coor_xyz

        d1    = calc.run.dots[1]
        dots1 = d1.coor_xyz

        max_cms, max_cms_per_atom, calc2 = calculate_maximum_possible_contact_ms(target_xyz, target_radii)
        # print('Max CMS:', max_cms)
        cms_results.append((file, cms, max_cms))

df = pd.DataFrame(cms_results, columns=['PDB', 'CMS', 'Max_CMS'])
df

,PDB,CMS,Max_CMS
0,2rol_0001.pdb,259.635099,1198.921690
1,5m5r_0014.pdb,304.272898,769.203708
2,2fgr_0005.pdb,198.491122,735.649712
3,4nuu_0006.pdb,379.099373,1052.829508
4,3emw_0013.pdb,236.786745,631.875297
...,...,...,...
165,1ywi_0001.pdb,194.438812,540.607745
166,4x2h_0011.pdb,268.558497,590.593518
167,6f0w_0007.pdb,612.809089,1605.769338
168,5di8_0015.pdb,410.363707,1043.251940


In [6]:
df.to_csv("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/scale/cms_results.csv", index=False)